In [1]:
from llama_cpp import Llama, llama_supports_gpu_offload 
import pandas as pd      
import os
import json

In [2]:
print("GPU offload supported:", llama_supports_gpu_offload())

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce RTX 3060 Laptop GPU, compute capability 8.6, VMM: yes


GPU offload supported: True


In [3]:
# llm = Llama(
#         model_path="../llama.cpp/Qwen3-1.7B-Q8_0.gguf",
#         n_ctx=32768,
#         n_gpu_layers=-1             
#         )  


llm_recipe1 = Llama(
        model_path="../llama.cpp/Qwen3-1.7B-Q8_0.gguf",
        # lora_path="../llama.cpp/Qwen3-1.7b-lora-recipe1-q8_0.gguf",
        lora_path="../llama.cpp/qwen3-1.7b-lora-recipe1-f16.gguf",
        n_ctx=32768,
        n_gpu_layers=-1             
        )  


# llm_in_use = llm_recipe1

llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GeForce RTX 3060 Laptop GPU) - 5120 MiB free
llama_model_loader: loaded meta data with 35 key-value pairs and 311 tensors from ../llama.cpp/Qwen3-1.7B-Q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen3 1.7B Hf
llama_model_loader: - kv   3:                           general.finetune str              = hf
llama_model_loader: - kv   4:                           general.basename str              = Qwen3
llama_model_loader: - kv   5:                         general.size_label str              = 1.7B
llama_model_loader: - kv   6:                            gener

### quick data length check with tokenizer

In [ ]:
import json
import numpy as np


DATA_PATH = "api_call/dataset_sft.jsonl"   # your file with one JSON object per line

def render_messages_naive(messages):
    """
    Approximate serialization for token counting.
    (Not an exact Qwen chat template, but consistent and useful for length stats.)
    """
    parts = []
    for m in messages:
        role = m.get("role", "")
        content = m.get("content", "")
        if isinstance(content, str) and content:
            parts.append(f"{role}: {content}\n")
    return "".join(parts)

def tok_len(text: str) -> int:
    ids = llm_in_use.tokenize(text.encode("utf-8"), add_bos=True, special=True)
    return len(ids)

def percentile(sorted_vals, p):
    if not sorted_vals:
        return None
    i = int(round((p / 100.0) * (len(sorted_vals) - 1)))
    return sorted_vals[i]

lengths = []
too_long = 0
bad = 0

with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            messages = obj["messages"]
            if not isinstance(messages, list):
                raise ValueError("messages is not a list")
        except Exception:
            bad += 1
            continue

        text = render_messages_naive(messages)
        L = tok_len(text)
        lengths.append(L)
        if L > llm_in_use.n_ctx():
            too_long += 1

if not lengths:
    print("No valid samples found.")
else:
    s = sorted(lengths)
    print(f"samples: {len(s)}")
    print(f"bad_lines: {bad}")
    print(f"n_ctx: {llm_in_use.n_ctx()}")
    print(f"min: {s[0]}")
    print(f"mean: {np.mean(s):.2f}")
    print(f"p50: {percentile(s, 50)}")
    print(f"p90: {percentile(s, 90)}")
    print(f"p95: {percentile(s, 95)}")
    print(f"p99: {percentile(s, 99)}")
    print(f"max: {s[-1]}")
    print(f"> n_ctx: {too_long} ({(too_long/len(s))*100:.2f}%)")
    print("\nTop 20 longest token counts:")
    for L in s[-20:][::-1]:
        print(L)

samples: 400
bad_lines: 0
n_ctx: 32768
min: 589
mean: 1097.09
p50: 1088
p90: 1380
p95: 1447
p99: 1552
max: 1655
> n_ctx: 0 (0.00%)

Top 20 longest token counts:
1655
1595
1589
1561
1552
1529
1525
1518
1515
1502
1487
1477
1471
1467
1464
1461
1458
1458
1452
1447


### smoke test

In [ ]:
response = llm_in_use.create_chat_completion(
    messages=[
        {"role": "assistant", "content": "You are a precise assistant."},
        {"role": "user", "content": "Do you know what are functional equations?"}
    ],
    temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
    max_tokens=None,
    stop=None#["<|im_end|>"],    
)
print(response["choices"][0]["message"]["content"])

llama_perf_context_print:        load time =     641.13 ms
llama_perf_context_print: prompt eval time =     640.43 ms /    26 tokens (   24.63 ms per token,    40.60 tokens per second)
llama_perf_context_print:        eval time =   51337.83 ms /   993 runs   (   51.70 ms per token,    19.34 tokens per second)
llama_perf_context_print:       total time =   55188.10 ms /  1019 tokens
llama_perf_context_print:    graphs reused =        961


<think>
Okay, the user is asking about functional equations. Let me start by defining what they are. Functional equations are equations where the unknowns are functions. So, instead of solving for a variable, we're solving for a function. For example, maybe something like f(x + y) = f(x) + f(y) which is Cauchy's equation. 

I should explain the general form of a functional equation. They usually involve finding a function that satisfies a certain condition for all values in a domain. The user might want to know how to solve them, but since they asked what they are, I should focus on the definition and examples.

Wait, the user might be a student who's new to functional equations. I should make sure to explain it in simple terms. Maybe mention that they're like algebraic equations but with functions instead of numbers. Also, note that the domain and codomain are important here.

Examples would help. Like the equation f(x) + f(-x) = 0, which is an odd function. Or the equation f(ax) = af

### prompts

In [ ]:


SYSTEM_PROMPT1 ="""
You are a mathematical problem STEP-BY-STEP PLANNER ONLY. You must outline actions without doing any calculations, derivations, simplifications, or symbolic transformations.

OUTPUT CONTRACT (MANDATORY)
- Output EXACTLY ONE plan block, delimited by the following sentinels:
  <<<PLAN>>>
  Steps:
  1) ...
  2) ...
  3) ...
- Plain text only.
- Each step is 1–2 sentences, actionable and specific.
- Use placeholders for ALL outcomes: <E1>, <E2>, <R1>, <C1>, <ANS>, etc.

HARD RESTRICTIONS
- DO NOT output anything before <<<PLAN>>>.
- DO NOT include templates, headings, examples, or explanations (e.g., “OUTPUT FORMAT”, “CORE BEHAVIOR”, “TEMPLATE”, etc.).
- DO NOT emit multiple plan blocks.
- DO NOT reveal any values, counts, divisors, factors, remainders, bases, simplified forms, or final answers.
- DO NOT show equations, arrows, or math operators: =, ≈, ≡, ⇒, →, +, -, ×, *, /, ÷, ^, %, mod, gcd, lcm, inequalities, or parentheses used for math grouping.
- DO NOT introduce new digits or number words not verbatim in the user’s problem. Placeholders like <E1> are allowed.
- DO NOT perform or describe base conversions, long division, polynomial/algebraic manipulation, modular reductions, factoring, listing divisors, numeric casework, or numeric comparisons.
- DO NOT perform any computation.

PLACEHOLDER POLICY
- Whenever a step would yield a concrete result, write: “Compute … and record it as <X>.”
- Use a new placeholder for each distinct outcome and reuse it when referenced later.

SELF-CHECK (BEFORE SENDING)
1) The output starts with <<<PLAN>>>.
2) There is exactly one line “Steps:” and exactly one numbered list of items.
3) No forbidden tokens appear: =, ≈, ≡, ⇒, →, +, -, ×, *, /, ÷, ^, %, mod, gcd, lcm, math parentheses.
4) No numbers beyond those verbatim in the user’s problem (placeholders exempt).
5) No computed values are shown—only instructions and placeholders.
If any check fails, silently rewrite to comply.
"""

SYSTEM_PROMPT1_extract ="""
You are a mathematical problem STEP-BY-STEP PLANNER ONLY. You must outline actions without doing any calculations, derivations, simplifications, or symbolic transformations.

OUTPUT CONTRACT (MANDATORY)
- Output EXACTLY ONE plan block, delimited by the following sentinels:
  <<<PLAN>>>
  Steps:
  1) ...
  2) ...
  3) ...
- First try to identify the required knowledge in order to solve the problem by extracting the covered subjets in the problem.
- Plain text only.
- Each step is 1–2 sentences, actionable and specific.
- Use placeholders for ALL outcomes: <E1>, <E2>, <R1>, <C1>, <ANS>, etc.

HARD RESTRICTIONS
- DO NOT output anything before <<<PLAN>>>.
- DO NOT include templates, headings, examples, or explanations (e.g., “OUTPUT FORMAT”, “CORE BEHAVIOR”, “TEMPLATE”, etc.).
- DO NOT emit multiple plan blocks.
- DO NOT reveal any values, counts, divisors, factors, remainders, bases, simplified forms, or final answers.
- DO NOT show equations, arrows, or math operators: =, ≈, ≡, ⇒, →, +, -, ×, *, /, ÷, ^, %, mod, gcd, lcm, inequalities, or parentheses used for math grouping.
- DO NOT introduce new digits or number words not verbatim in the user’s problem. Placeholders like <E1> are allowed.
- DO NOT perform or describe base conversions, long division, polynomial/algebraic manipulation, modular reductions, factoring, listing divisors, numeric casework, or numeric comparisons.
- DO NOT perform any computation.

PLACEHOLDER POLICY
- Whenever a step would yield a concrete result, write: “Compute … and record it as <X>.”
- Use a new placeholder for each distinct outcome and reuse it when referenced later.

SELF-CHECK (BEFORE SENDING)
1) The output starts with <<<PLAN>>>.
2) There is exactly one line “Steps:” and exactly one numbered list of items.
3) No forbidden tokens appear: =, ≈, ≡, ⇒, →, +, -, ×, *, /, ÷, ^, %, mod, gcd, lcm, math parentheses.
4) No numbers beyond those verbatim in the user’s problem (placeholders exempt).
5) No computed values are shown—only instructions and placeholders.
If any check fails, silently rewrite to comply.
"""




In [5]:
df = pd.read_parquet("AIME25.parquet")

In [6]:
example_task0 = df["problem"].iloc[0]

example_task1 = "Let the cubic $x^{3}-ax^{2}+bx-c=0$ have roots $r,s,t$ with $rs=6$, $rt=8$, $st=9$, and $r^{2}+s^{2}+t^{2}=38$. Find $a$, $b$, and $c$."

example_task3 = "Find the eigenvalues and unit eigenvectors $v_{1}, v_{2}$ of $A^{\mathsf T}A$. Then compute $u_{1}=\dfrac{A\,v_{1}}{\sigma_{1}}$, where $A=\begin{bmatrix}1&2\\3&6\end{bmatrix}$, $A^{\mathsf T}A=\begin{bmatrix}10&20\\20&40\end{bmatrix}$, and $A A^{\mathsf T}=\begin{bmatrix}5&15\\15&45\end{bmatrix}$. Verify that $u_{1}$ is a unit eigenvector of $A A^{\mathsf T}$. Complete the matrices $U$, $\Sigma$, and $V$ in the singular value decomposition $\,[u_{1}\ \ u_{2}]\,\begin{bmatrix}\sigma_{1}&0\\0&0\end{bmatrix}\,[v_{1}\ \ v_{2}]^{\mathsf T}$ of $A$."

example_task5 = "For an integer base $b>9$, let $N_b$ be the three-digit base-$b$ integer whose digits (in order) are $(b-1)$, $(b-2)$, and $3$; that is, $N_b=(b-1)\cdot b^{2}+(b-2)\cdot b+3$. Determine the sum of all bases $b$ such that $(b+1)$ divides $N_b$ and, additionally, the base-$b$ integer $1(b-3)(b-4)_b$ is divisible by $(b-2)$."

example_task6 = "Prove that if $A$ is full rank, then for the optimization problem $\min_{x}\,\lVert A x - b \rVert_{2}$ we have $x = V \Sigma^{-1} U^{T} b$, where $U$, $\Sigma$, and $V$ are from the SVD of $A$."

example_task7 = "Find all ordered pairs of primes $(p,q)$ with $p<q$ such that $p \mid (q^{2}+1)$ and $q \mid (p^{3}+1)$. Prove that your list is complete and justify each implication."

example_task8 = "Let $triangle ABC$ be acute with circumradius $R$, inradius $r$, circumcenter $O$, incenter $I$, and $a=BC$, $b=CA$, $c=AB$. Let the tangents to the circumcircle at $B$ and $C$ meet at $T$. Prove that $T I\perp A I$ and that $\dfrac{A I}{I T}=\dfrac{b c}{(b+c-a)\,2R}$ expressed purely in terms of $a$, $b$, $c$, and $R$. Then, using only triangle identities and standard power/area relations, deduce the formula $(O I)^{2}=R\,(R-2 r)$ as a corollary."

example_task9 = "For a real parameter $a>0$, define $I(a)=\int_{0}^{\infty}\left(\frac{\ln(1+a x)}{x(1+x)}-\frac{a}{1+a x}\right)\,dx$. Prove that $I(a)$ converges for all $a>0$, obtain a closed form for $I(a)$ in terms of $a$, and determine the unique value $a neq 1$ such that $I(a)=0$."


example_task2 = "A deck has $8$ black and $7$ red cards. Cards are drawn without replacement until $3$ reds have appeared. What is the probability that exactly $6$ draws are needed?"

example_task4 = "An urn initially contains $5$ red, $4$ blue, and $3$ green balls. Three balls are drawn uniformly at random without replacement. If at least two of the three are red, then $3$ blue balls are added to the urn; if exactly one is red, then $2$ red and $1$ green are added; otherwise, $4$ red are added. After the addition, one more ball is drawn. What is the probability that this fourth ball is red?"

example_task10 = "A factory produces bolts on two lines. Line A makes $420$ bolts per hour with defect probability $0.03$ per bolt; Line B makes $280$ bolts per hour with defect probability $0.05$ per bolt. During a $30$-minute window, a quality engineer randomly samples $12$ bolts uniformly from all bolts produced in that window. If the sample contains at least $3$ defective bolts, the engineer shuts down the line that contributed the larger number of defective bolts in the sample (break ties by shutting down Line B). What is the probability that Line B is shut down?"

example_task11 = "A bank offers two repayment options for a loan of €$18{,}000$ over $12$ months. Plan 1 charges a fixed monthly interest rate of $1.2%$ on the remaining balance and requires equal monthly payments. Plan 2 charges $0.9%$ monthly interest for the first $6$ months and $1.5%$ for the last $6$ months, and requires payments of €$1{,}200$ for months $1$–$6$ and a constant payment amount €$P$ for months $7$–$12$ that fully amortizes the loan. What is the value of €$P$, and by how many euros is the total paid under Plan 2 larger or smaller than under Plan 1?"

example_task12 = "A right circular cylindrical tank has radius $3.5$ m and height $12$ m. It is initially filled with water to a height of $9$ m. Water is pumped out at a constant rate of $0.18$ m$^3$/min for $40$ minutes. Immediately after, a different pump adds water at a constant rate of $0.12$ m$^3$/min while a leak simultaneously drains water at a rate proportional to the current water height: $0.015,h$ m$^3$/min when the height is $h$ meters. After $60$ minutes of this second phase, what is the water height in the tank (in meters)?"

example_task13 = "How many integers $n$ with $1 \le n \le 10^8$ satisfy all of the following: (i) $n \equiv 17 \pmod{48}$; (ii) $n \equiv 5 \pmod{75}$; (iii) $n$ is a multiple of $9$; and (iv) in decimal, $n$ has exactly $7$ digits and its last two digits form a number divisible by $4$"

example_task14 = "A retailer sells $3$ products: A, B, and C. In a given week, the store receives $120$ units of A, $90$ units of B, and $60$ units of C. Customer demand that week is random: the number of requested units for A, B, C are independent and distributed as $D_A\sim\text{Poisson}(110)$, $D_B\sim\text{Poisson}(95)$, and $D_C\sim\text{Poisson}(70)$. Any unmet demand is lost (no backorders). Unit profits are €$4.20$ for A, €$5.10$ for B, and €$7.80$ for C. At the end of the week, leftover inventory is salvaged at €$0.60$ per unit for A, €$0.80$ per unit for B, and €$1.20$ per unit for C. Additionally, if the total number of units sold across all products is at least $230$, the store earns a bonus of €$150$; otherwise there is no bonus. What is the expected total profit for the week?" 

# -------------------------
# Cluster 1: Adversarial language and specification ambiguity
# -------------------------

cluster1_1 = "A number $N$ is written in base $10$ with no leading zeros. Exactly one of the following statements about $N$ is true: (i) $N$ is divisible by $6$; (ii) the sum of the digits of $N$ is divisible by $9$; (iii) $N$ leaves remainder $1$ when divided by $4$. Given that $1000 \\le N \\le 9999$ and that $N$ is divisible by $5$, how many possible values of $N$ are there?"

cluster1_2 = "A sequence $(a_n)$ of real numbers satisfies the following rule: for every integer $n \\ge 1$, either $a_{n+1} = 2a_n - 1$ or $a_{n+1} = 2a_n + 1$, and the choice at each step is made so that among the three numbers $a_n, a_{n+1}, a_{n+2}$ exactly one is an integer. If $a_1 = \\tfrac{1}{3}$, determine whether $a_{100}$ must be rational, must be irrational, or can be either depending on the choices."

cluster1_3 = "A medical test returns either Positive or Negative. The disease prevalence is $0.02$. The test has sensitivity $0.93$ and specificity $0.96$. A clinic repeats the test independently a second time only if the first result is Positive. The clinic declares a person \"Infected\" if and only if at least one of the performed tests is Positive. What is the probability that a randomly selected person declared Infected is actually infected?"

cluster1_4 = "A bag contains $10$ slips labeled with integers. You are told that at least $6$ slips have distinct labels and that the sum of all $10$ labels is $17$. You draw $3$ slips uniformly at random without replacement. You win if the maximum of the three labels is strictly greater than the sum of the other two labels. What is the maximum possible value of your winning probability over all labelings consistent with the information?"

# -------------------------
# Cluster 2: Symbolic structure and invariant-heavy algebra
# -------------------------

cluster2_1 = "A function $f : \\mathbb{R} \\to \\mathbb{R}$ satisfies $f(x+y) + f(x-y) = 2f(x)f(y)$ for all real $x,y$, with $f(0) = 1$. The function is continuous at $0$ and satisfies $f(1) = \\tfrac{3}{2}$. Determine the value of $f(2)$."

cluster2_2 = "Three integers $(x,y,z)$ are written on a board. In one move, you may replace exactly one of the numbers by the sum of the other two, so $(x,y,z)$ may become $(y+z,y,z)$, $(x,x+z,z)$, or $(x,y,x+y)$. Starting from $(1,1,2)$, is it possible to reach a state in which all three numbers are multiples of $7$? Justify your answer."

cluster2_3 = "Let $(a_n)$ be defined by $a_1 = 1$, $a_2 = 3$, and $a_{n+2} = 4a_{n+1} - 4a_n$ for all integers $n \\ge 1$. Find all integers $n$ with $1 \\le n \\le 50$ for which $a_n$ is a perfect square."

cluster2_4 = "Let $x,y,z > 0$ satisfy $xyz = 1$. Prove or disprove the inequality $(x+1)(y+1)(z+1) \\ge 8$, and determine all cases of equality if it holds or describe all counterexample conditions if it does not."

# -------------------------
# Cluster 3: Geometric or spatial reasoning with implicit constraints
# -------------------------

cluster3_1 = "In triangle $ABC$, the side lengths are $AB = 13$, $BC = 14$, and $CA = 15$. Point $D$ lies on segment $BC$ such that $BD = 6$. Let the circumcircle of triangle $ABD$ intersect segment $AC$ again at a point $E \\ne A$. Find the length $AE$."

cluster3_2 = "A circle $\\omega$ has center $O$ and radius $10$. A chord $AB$ of $\\omega$ has length $12$. Point $P$ lies on the minor arc $AB$. The tangents to $\\omega$ at $A$ and $B$ meet at point $T$. Given that line $TP$ intersects segment $AB$ at its midpoint, determine the measure of $\\angle AOB$ in degrees."

cluster3_3 = "In three-dimensional space, consider the tetrahedron with vertices $A=(0,0,0)$, $B=(6,0,0)$, $C=(0,8,0)$, and $D=(0,0,3)$. Let $M$ be the point on segment $BD$ such that $BM:MD = 2:1$. Find the distance from point $M$ to the plane through points $A$, $C$, and $D$."

cluster3_4 = "A right circular cone has base radius $9$ and height $12$. A plane cuts the cone and intersects the base circle in a chord of length $12$, producing an elliptical cross-section. The plane is perpendicular to the base, and its line of intersection with the base passes through the center of the base circle. Determine the area of the elliptical cross-section."

cluter1 = [cluster1_1, cluster1_2, cluster1_3, cluster1_4]
cluster2 = [cluster2_1, cluster2_2, cluster2_3, cluster2_4]
cluster3 = [cluster3_1, cluster3_2, cluster3_3, cluster3_4]

example_tasks = [example_task0,example_task1,example_task3,example_task5,example_task6,example_task7,example_task8,example_task9]


long_numerical_tasks = [example_task2, example_task4, example_task10, example_task11, example_task12, example_task13, example_task14]

<>:5: SyntaxWarning: invalid escape sequence '\m'
<>:7: SyntaxWarning: invalid escape sequence '\c'
<>:9: SyntaxWarning: invalid escape sequence '\m'
<>:11: SyntaxWarning: invalid escape sequence '\m'
<>:13: SyntaxWarning: invalid escape sequence '\p'
<>:15: SyntaxWarning: invalid escape sequence '\i'
<>:28: SyntaxWarning: invalid escape sequence '\l'
<>:30: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\m'
<>:7: SyntaxWarning: invalid escape sequence '\c'
<>:9: SyntaxWarning: invalid escape sequence '\m'
<>:11: SyntaxWarning: invalid escape sequence '\m'
<>:13: SyntaxWarning: invalid escape sequence '\p'
<>:15: SyntaxWarning: invalid escape sequence '\i'
<>:28: SyntaxWarning: invalid escape sequence '\l'
<>:30: SyntaxWarning: invalid escape sequence '\s'
C:\Users\pouri\AppData\Local\Temp\ipykernel_4372\3163560408.py:5: SyntaxWarning: invalid escape sequence '\m'
  example_task3 = "Find the eigenvalues and unit eigenvectors $v_{1}, v_{2}$ of 

### error analysis

In [ ]:
import os
import json
import re

# --- Helper: safe filenames from task strings ---
def slugify(value: str, max_length: int = 80) -> str:
    value = value.strip()
    value = re.sub(r"[\/\:\*\?\"<>\|\x00-\x1F]", "_", value)  # Replace path separators and control chars
    value = re.sub(r"\s+", " ", value)                        # Collapse spaces
    value = value[:max_length].rstrip()
    return value or "task"

# Accept lines that ALREADY look like steps. We DO NOT strip their markers.
STEP_LINE_RE = re.compile(
    r"""^\s*(
        (?:Step\s*\d+\s*[:\.\-\)]\s*)     # "Step 1:", "Step1)", etc.
        |(?:\d{1,2}[.\)]\s+)              # "1. ", "2) "
        |(?:[-\*\u2022]\s+)               # "-", "*", "• "
    ).+""",
    re.IGNORECASE | re.VERBOSE
)

STEPS_SECTION_RE = re.compile(r'^\s*Steps\s*:\s*$', re.IGNORECASE)

def extract_steps_verbatim(text: str):
    """
    Extract step lines verbatim (keep numbering/bullets).
    Priority:
      1) If a 'Steps:' header exists, collect subsequent lines that match step format,
         stopping at the first blank line block (or end).
      2) Else, collect ANY lines in the whole text that match step format.
      3) If none found, fall back to all non-empty lines (verbatim).
    """
    if not text:
        return []

    # Normalize line endings
    lines = text.replace("\r\n", "\n").replace("\r", "\n").split("\n")

    # 1) Try to find a 'Steps:' section and gather from there
    steps_after_header = []
    in_steps = False
    for i, line in enumerate(lines):
        if not in_steps:
            if STEPS_SECTION_RE.match(line):
                in_steps = True
            continue
        # Once inside 'Steps:' block, collect step-like lines until blank block ends it
        if line.strip() == "":
            # stop when we hit the first empty line AFTER we've started collecting
            if steps_after_header:
                break
            else:
                # allow initial blank lines right after 'Steps:' if any
                continue
        if STEP_LINE_RE.match(line):
            steps_after_header.append(line.rstrip())
        else:
            # If we encounter a non-step line after collecting at least one, end the block
            if steps_after_header:
                break
            # If we haven't collected any step yet, keep scanning (some models place notes)
            continue

    if steps_after_header:
        return steps_after_header

    # 2) Else, collect any step-like lines across the whole text
    any_steps = [ln.rstrip() for ln in lines if STEP_LINE_RE.match(ln)]
    if any_steps:
        return any_steps

    # 3) Fallback: return all non-empty lines verbatim
    return [ln.strip() for ln in lines if ln.strip()]

# --- Ensure output directory exists ---
OUTPUT_DIR = "example_tasks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Core loop ---
for j, task in enumerate(example_tasks):
    responses = {}
    for i in range(5):
        response = llm_in_use.create_chat_completion(
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT1},
                {
                    "role": "user",
                    "content": "/no_think \nDO NOT SOLVE FOR FINAL ANSWER, JUST PLAN. \n Problem: \n" + task,
                },
            ],
            temperature=0.60,
            top_p=0.95,
            top_k=20,
            min_p=0.0,
            max_tokens=None,
            stop=["<|im_end|>", "<<<END>>>"],
        )

        raw_text = response["choices"][0]["message"]["content"]

        # Remove lines containing the specified substrings
        filtered_lines = []
        for line in raw_text.splitlines():
            if ('llama_perf_context_print:' in line) or ('Llama.generate' in line):
                continue
            filtered_lines.append(line)
        cleaned_text = "\n".join(filtered_lines).strip()

        # Extract steps VERBATIM (no added prefixes or re-numbering)
        steps = extract_steps_verbatim(cleaned_text)

        # Save as an array; each element is exactly what the model wrote for that step line
        responses[f"response{i+1}"] = steps

    # Write per-task JSON file: { "<task_string>": { "response1": [...], ... } }
    data = {task: responses}

    filename = f"example_task{j}.json"  # or use: slugify(task) + ".json"
    out_path = os.path.join(OUTPUT_DIR, filename)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


NameError: name 'SYSTEM_PROMPT1' is not defined

### single inference

In [4]:
SYSTEM_PROMPT_STRONG = """
You are a human-like math problem PLANNER ONLY. Do not solve. Do not compute/derive/simplify. No numeric results.

Task: extract ONLY what is necessary to reach the asked-for quantity <ANS>.
Omit anything not used downstream.

Output exactly one plain-text block:
<<<PLAN>>>
Goal: <ANS> = (what the problem asks, in words)
Unknowns:
- <U1>=...
- <U2>=...   
(only true unknowns / decision outcomes)
Core constraints:
- <C1>: ...
- <C2>: ...
Dependency skeleton:
- <R1> from <C?> and <U?>
- <R2> from <C?> and <R1>
Plan:
- Identify the minimum set of intermediates needed for <ANS>.
- Compute ... -> <R1>.
- Compute ... -> <R2>.
- Combine intermediates to express <ANS>. (still no execution)

<<<END>>>

Rules:
- Do NOT create placeholders for fixed givens; refer to them as “given in the statement”.
- Do NOT name standard sub-parameters unless they are required intermediates; use <Rk> instead.
- No equations or operator symbols; describe relations in words.
- Nothing outside the block.
"""

In [7]:
example = "A deck has 19 red and 29 black cards. You draw 3 cards without replacement and only learn that the number of reds is in the set {1,2,3}. Then you draw 7 more cards from the remaining deck. What is the probability that the second batch contains at least 3 red given this partial information?"
example_valid = "A recipe uses 576 grams ingredient X, 219 mL ingredient Y, and 381 grams ingredient Z to produce 13 servings. When scaling to 36 servings, ingredient X experiences 15.77% yield loss during cooking (so you must start with more), and ingredient Y must be measured in multiples of 25 mL (rounded up). If the pre-discount total cost exceeds 449.0, a discount 33.19% applies only to ingredient Z. Given unit costs cx=0.138 per gram, cy=0.009 per mL, cz=0.165 per gram, compute total cost and final amounts of each ingredient used."
QUESTION_TEXT = """
A triangle has sides a,b,c with a:b = 5:2 and perimeter 202. Its area is 863.588. If the altitude to side a is less than 31.965, the triangle is scaled uniformly by factor 1.789. Find the final side lengths and the final inradius.
"""


for _ in range(1):
    response = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_STRONG},
            {
                "role": "user",
                "content": "/no_think \nDEVISE A PLAN FOR THIS PROBLEM. \n Problem: \n" + example_valid,
            },
        ],
        temperature=0.05,
        top_p=0.99,
        top_k=20,
        min_p=0.0,
        max_tokens=2000,
        stop=["<|im_end|>",]# "<<<END>>>"],
    )

    print("📍")
    print(response["choices"][0]["message"]["content"])
    print("########################################")
    filename = f"example_task.json"  # or use: slugify(task) + ".json"
    out_path = os.path.join(".", filename)
    with open(out_path, "a", encoding="utf-8") as f:
        json.dump(response["choices"][0]["message"]["content"], f, ensure_ascii=False, indent=2)


Llama.generate: 283 prefix-match hit, remaining 161 prompt tokens to eval
llama_perf_context_print:        load time =     977.20 ms
llama_perf_context_print: prompt eval time =     563.54 ms /   161 tokens (    3.50 ms per token,   285.69 tokens per second)
llama_perf_context_print:        eval time =    3319.93 ms /   305 runs   (   10.88 ms per token,    91.87 tokens per second)
llama_perf_context_print:       total time =    4326.40 ms /   466 tokens
llama_perf_context_print:    graphs reused =        294


📍
<think>

</think>

<<<PLAN>>>  
Goal: <ANS> = (total cost and final amounts of each ingredient used)  
Unknowns:  
- <U1>=final amount of ingredient X in grams  
- <U2>=final amount of ingredient Y in mL  
- <U3>=final amount of ingredient Z in grams  
Core constraints:  
- <C1>: ingredient X is scaled with 15.77% yield loss  
- <C2>: ingredient Y is measured in multiples of 25 mL (rounded up)  
- <C3>: pre-discount total cost exceeds 449.0  
- <C4>: discount 33.19% applies only to ingredient Z  
Dependency skeleton:  
- <R1> from <C1> and <U1>  
- <R2> from <C2> and <U2>  
- <R3> from <C3> and <R1>  
- <R4> from <C4> and <R3>  
Plan:  
- Compute final amount of ingredient X using <C1> and <U1>  
- Compute final amount of ingredient Y using <C2> and <U2>  
- Compute pre-discount total cost using <R1> and <R3>  
- Apply discount 33.19% to <R3> using <C4>  
- Combine results to express <ANS>
########################################


In [5]:
example = "A deck has 19 red and 29 black cards. You draw 3 cards without replacement and only learn that the number of reds is in the set {1,2,3}. Then you draw 7 more cards from the remaining deck. What is the probability that the second batch contains at least 3 red given this partial information?"
example_valid ="A recipe uses 576 grams ingredient X, 219 mL ingredient Y, and 381 grams ingredient Z to produce 13 servings. When scaling to 36 servings, ingredient X experiences 15.77% yield loss during cooking (so you must start with more), and ingredient Y must be measured in multiples of 25 mL (rounded up). If the pre-discount total cost exceeds 449.0, a discount 33.19% applies only to ingredient Z. Given unit costs cx=0.138 per gram, cy=0.009 per mL, cz=0.165 per gram, compute total cost and final amounts of each ingredient used."
QUESTION_TEXT = """
A triangle has sides a,b,c with a:b = 5:2 and perimeter 202. Its area is 863.588. If the altitude to side a is less than 31.965, the triangle is scaled uniformly by factor 1.789. Find the final side lengths and the final inradius.
"""

for _ in range(1):
    response = llm_recipe1.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_STRONG},
            {
                "role": "user","content": "/no_think" + example_valid,},
        ],
        temperature=0.05,
        top_p=0.99,
        top_k=20,
        min_p=0.0,
        max_tokens=None,
        stop=["<|im_end|>",]# "<<<END>>>"],
    )

    print("📍")
    print(response["choices"][0]["message"]["content"])
    print("########################################")
    filename = f"example_task.json"  # or use: slugify(task) + ".json"
    out_path = os.path.join(".", filename)
    with open(out_path, "a", encoding="utf-8") as f:
        json.dump(response["choices"][0]["message"]["content"], f, ensure_ascii=False, indent=2)


llama_perf_context_print:        load time =    1209.50 ms
llama_perf_context_print: prompt eval time =    1208.85 ms /   430 tokens (    2.81 ms per token,   355.71 tokens per second)
llama_perf_context_print:        eval time =    4377.26 ms /   320 runs   (   13.68 ms per token,    73.11 tokens per second)
llama_perf_context_print:       total time =    5976.49 ms /   750 tokens
llama_perf_context_print:    graphs reused =        309


📍
<think>

</think>

<<<PLAN>>>  
Goal: <ANS> = (total cost and final amounts of each ingredient used)  
Unknowns:  
- <U1>=final amount of ingredient X in grams  
- <U2>=final amount of ingredient Y in mL  
- <U3>=final amount of ingredient Z in grams  
Core constraints:  
- <C1>: ingredient X is scaled by a factor of 13/36, but with a 15.77% yield loss, so the amount must be adjusted to account for this loss  
- <C2>: ingredient Y is measured in multiples of 25 mL, rounded up  
- <C3>: ingredient Z is scaled by a factor of 13/36, but with a 33.19% discount applied only if the pre-discount total cost exceeds 449.0  
Dependency skeleton:  
- <R1> from <C1> and <U1>  
- <R2> from <C2> and <U2>  
- <R3> from <C3> and <R2>  
Plan:  
- Identify the minimum set of intermediates needed for <ANS>  
- Compute <R1> from <C1> and <U1>  
- Compute <R2> from <C2> and <R1>  
- Compute <R3> from <C3> and <R2>  
- Combine intermediates to express <ANS> (still no execution)
###########################